# Vector Search

This notebook performs semantic search over the indexed CRMP chunks. Queries are embedded with the same `bge-m3` model used during indexing and matched against the Qdrant collection by cosine similarity.

It also demonstrates how retrieved chunks can be assembled into grounded context for a future answer-generation stage.


## 1. Import the required clients

Load HTTP support for Ollama and the Qdrant client for vector retrieval.


In [1]:
import requests

from qdrant_client import QdrantClient

## 2. Configure local services

Define the Qdrant collection, Ollama endpoint, and embedding model.


In [2]:
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "crmp_bge_m3"

OLLAMA_URL = "http://localhost:11434/api/embed"
EMBEDDING_MODEL = "bge-m3"

## 3. Connect to Qdrant

Initialize the client and confirm that the expected collection service is reachable.


In [3]:
client = QdrantClient(
    url=QDRANT_URL
)

print(client.get_collections())

collections=[CollectionDescription(name='crmp_bge_m3')]


## 4. Define query embedding

Create a helper that converts a natural-language query into a `bge-m3` vector.


In [4]:
def get_embedding(
    text,
    model=EMBEDDING_MODEL
):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    # Stop immediately if Ollama cannot produce a valid query embedding.
    response.raise_for_status()

    return response.json()["embeddings"][0]


## 5. Embed a sample query

Validate the query pipeline and confirm that vector dimensions match the collection.


In [5]:
query = "Quais são as regras relativas ao estacionamento?"

# Use the same model as indexing so query and document vectors share a space.
query_vector = get_embedding(query)

print(f"Dimensão do embedding: {len(query_vector)}")


Dimensão do embedding: 1024


## 6. Define vector search

Combine query embedding and Qdrant similarity search in one reusable function.


In [6]:
def vector_search(
    query,
    limit=5
):
    query_vector = get_embedding(query)

    # Qdrant ranks the closest stored chunk vectors to the query vector.
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=limit,
        with_payload=True
    )

    return results.points


## 7. Run an initial search

Retrieve the top five chunks for a parking-related question.


In [7]:
results = vector_search(
    "Quais são as regras relativas ao estacionamento?",
    limit=5
)

## 8. Inspect raw search results

Display ranking, scores, article metadata, and text returned by Qdrant.


In [8]:
# Enumerate from one to present results as a human-readable ranking.
for position, result in enumerate(
    results,
    start=1
):
    payload = result.payload

    print("=" * 100)

    print(
        f"Resultado #{position}"
    )

    print(
        f"Score: {result.score:.4f}"
    )

    print(
        f"Artigo: {payload.get('article')}"
    )

    print(
        f"Epígrafe: {payload.get('article_title')}"
    )

    print(
        f"Parte: {payload.get('part')}"
    )

    print(
        f"Páginas: "
        f"{payload.get('page_start')} - "
        f"{payload.get('page_end')}"
    )

    print()

    print(payload.get("text"))

    print()


Resultado #1
Score: 0.6450
Artigo: D-3/24.º
Epígrafe: Estacionamento e paragem permitida
Parte: D
Páginas: 179 - 180

Artigo D-3/24.º
Estacionamento e paragem permitida
1 – O estacionamento ou a paragem devem fazer-se nos locais especialmente destinados
a esse fim e da forma indicada na respetiva sinalização, devendo processar-se o mais
próximo possível do limite direito da faixa de rodagem, paralelamente a esta e no sentido
da marcha, salvo se, por meio de sinalização, a disposição ou a geometria indicarem outra
forma.
2 – O condutor, ao deixar o veículo estacionado, deve guardar os intervalos indispensáveis
para manobra de saída de outros veículos ou de ocupação de espaços vagos.
3 – O estacionamento deve processar-se de forma a permitir a normal fluidez do trânsito,
não impedindo nem dificultando o acesso à propriedade privada nem prejudicando a
circulação de peões.
Gestão do Espaço Público
4 – Nos parques e zonas de estacionamento, os condutores devem estacionar de forma a
ocupar a

## 9. Create a readable result viewer

Wrap search and formatted output for quick interactive exploration.


In [9]:
def show_results(
    query,
    limit=5
):
    results = vector_search(
        query,
        limit=limit
    )

    print(f"QUERY: {query}")
    print()

    for position, result in enumerate(
        results,
        start=1
    ):
        payload = result.payload

        print(
            f"{position}. "
            f"{payload.get('article')} - "
            f"{payload.get('article_title')}"
        )

        print(
            f"   Score: {result.score:.4f}"
        )

        print(
            f"   Página: "
            f"{payload.get('page_start')}"
        )

        print()

## 10. Search for resident parking permits

Test retrieval with a specific entitlement question.


In [10]:
show_results(
    "Quem pode pedir uma avença de residente?"
)

QUERY: Quem pode pedir uma avença de residente?

1. D-6/15.º - Avença de residente
   Score: 0.7078
   Página: 252

2. D-6/16.º - Condições de atribuição da avença residente
   Score: 0.6910
   Página: 253

3. D-6/18.º - Direitos e deveres do titular da avença de residente
   Score: 0.6603
   Página: 254

4. D-6/17.º - Validade da avença de residente
   Score: 0.6373
   Página: 254

5. D-3/64.º - Avenças e títulos de estacionamento nos parques de estacionamento municipais
   Score: 0.5654
   Página: 189



## 11. Search for public-space rules

Evaluate retrieval on a different regulatory topic.


In [11]:
show_results(
    "Quais são as regras para ocupação do espaço público?"
)

QUERY: Quais são as regras para ocupação do espaço público?

1. D-1/7.º-B - Condições aplicáveis à ocupação do espaço público
   Score: 0.6933
   Página: 122

2. D-1/7.º-A - Proibições aplicáveis à ocupação do espaço público
   Score: 0.6576
   Página: 121

3. D-1/32.º - Condições gerais
   Score: 0.6534
   Página: 138

4. 1.º-A - Condições gerais da ocupação do espaço público
   Score: 0.6466
   Página: 478

5. C-2/9.º - Ocupação do espaço verde
   Score: 0.6416
   Página: 77



## 12. Search for application requirements

Test whether the system retrieves procedural document requirements.


In [12]:
show_results(
    "Que documentos são necessários para apresentar um requerimento?"
)

QUERY: Que documentos são necessários para apresentar um requerimento?

1. A-2/4.º - Requisitos comuns do requerimento
   Score: 0.6704
   Página: 27

2. E-7/8.º-A - Requerimento de candidatura
   Score: 0.6645
   Página: 333

3. 24.º - Condições de instalação e manutenção de tapetes ou equiparados
   Score: 0.6234
   Página: 486

4. 24.º - Condições de instalação e manutenção de tapetes ou equiparados
   Score: 0.6232
   Página: 486

5. D-3/55.º - Requerimento de aprovação
   Score: 0.6081
   Página: 185



## 13. Search for data-protection rules

Evaluate a query from another legal subject area.


In [13]:
show_results(
    "Existem regras relativas à proteção de dados pessoais?"
)

QUERY: Existem regras relativas à proteção de dados pessoais?

1. A-1/7.º - Proteção de Dados
   Score: 0.6490
   Página: 24

2. E-7/13.º - Registo
   Score: 0.5256
   Página: 337

3. A-2/4.º - Requisitos comuns do requerimento
   Score: 0.5054
   Página: 27

4. C-3/15.º - Normas de circulação
   Score: 0.4850
   Página: 114

5. E-7/39.º - Emissão da licença
   Score: 0.4807
   Página: 350



## 14. Define context retrieval

Convert ranked Qdrant results into a compact text block for an LLM prompt.


In [14]:
def retrieve_context(
    query,
    limit=5
):
    results = vector_search(
        query,
        limit=limit
    )

    # Preserve article references so generated answers can cite retrieved evidence.
    context = []

    for result in results:
        payload = result.payload

        context.append({
            "score": result.score,
            "article": payload.get("article"),
            "article_title": payload.get("article_title"),
            "page_start": payload.get("page_start"),
            "page_end": payload.get("page_end"),
            "text": payload.get("text")
        })

    return context


## 15. Build context for a sample question

Generate retrieval context for a municipal application query.


In [15]:
context = retrieve_context(
    "Como devo apresentar um requerimento?"
)

context

[{'score': 0.6517252,
  'article': 'A-2/3.º',
  'article_title': 'Forma de apresentação dos requerimentos',
  'page_start': 27,
  'page_end': 27,
  'text': 'Artigo A-2/3.º\nForma de apresentação dos requerimentos\n1 – Sem prejuízo do disposto nos números seguintes, os requerimentos devem ser\napresentados por escrito ou, nos casos em que a lei o admita, verbalmente, através dos\ncanais de atendimento disponibilizados pelo Município e divulgados no respetivo sítio\ninstitucional, ou através do Balcão Único Eletrónico, quando aplicável, ou através de outras\nplataformas determinadas legalmente.\n2 – Os requerimentos relativos aos procedimentos urbanísticos devem ser apresentados e\ninstruídos com recurso aos meios eletrónicos disponibilizados pelo Município.\n3 – (Revogado.)\n4 – De forma a garantir a igualdade no acesso aos serviços da Administração, o Município\ndo Porto disponibiliza um serviço de atendimento assistido aos munícipes para a\nsubmissão dos requerimentos por meios eletró